# MedSIGHT — VQA-RAD Inference Demo

This notebook walks through:

1. Loading a MedSIGHT model from `configs/model.yaml`.
2. Querying it on a single VQA-RAD image.
3. Running batch inference over the full VQA-RAD test split and writing
   `answers.jsonl` in the same format the scoring scripts expect.

For full-dataset evaluation use `python -m llava.eval.inference` (or
`scripts/run_inference.sh`) instead — this notebook is meant for exploration.

In [1]:
import os, sys
# REPO_ROOT = os.environ.get('MEDSIGHT_REPO', '/path/to/RegTok/RegLLM')
REPO_ROOT = os.environ.get('MEDSIGHT_REPO', '/home/avc6555/research/MedSight/RegTok/RegLLM')
sys.path.insert(0, REPO_ROOT)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

from llava.eval.chatbot import RegLLMChatbot

MODEL_CONFIG = os.path.join(REPO_ROOT, 'llava/eval/configs/model.yaml')
bot = RegLLMChatbot.from_config(MODEL_CONFIG, device='cuda')

/data/aofei/conda/env/med_edit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/aofei/conda/env/med_edit/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading model from /data/aofei/output/MedSight/1110_full_instruct_71k_nosep
Loaded 576 codebook token ids from /data/aofei/output/MedSight/1110_full_instruct_71k_nosep/added_tokens.json
Initializing RegSegForCausalLM with config: LlavaQwenConfig {
  "architectures": [
    "RegSegForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "freeze_mm_mlp_adapter": false,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "image_aspect_ratio": "square",
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "max_position_embeddings": 40960,
  "max_window_layers": 36,
  "mm_hidden_size": 1024,
  "mm_patch_merge_type": "flat",
  "mm_projector_lr": null,
  "mm_projector_type": "mlp2x_gelu",
  "mm_use_im_patch_token": false,
  "mm_use_im_start_end": false,
  "mm_vision_select_feature": "patch",
  "mm_vision_select_layer": -2,
  "mm_vision_tower": "/qumulo/shared_data/aofei_summer/CLIPs/unimed_clip_vit_l1

Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.29s/it]
Some weights of the model checkpoint at /data/aofei/output/MedSight/1110_full_instruct_71k_nosep were not used when initializing RegSegForCausalLM: ['model.vision_tower.vision_tower.image_encoder.class_embedding', 'model.vision_tower.vision_tower.image_encoder.conv1.weight', 'model.vision_tower.vision_tower.image_encoder.ln_post.bias', 'model.vision_tower.vision_tower.image_encoder.ln_post.weight', 'model.vision_tower.vision_tower.image_encoder.ln_pre.bias', 'model.vision_tower.vision_tower.image_encoder.ln_pre.weight', 'model.vision_tower.vision_tower.image_encoder.positional_embedding', 'model.vision_tower.vision_tower.image_encoder.proj', 'model.vision_tower.vision_tower.image_encoder.transformer.resblocks.0.attn.in_proj_bias', 'model.vision_tower.vision_tower.image_encoder.transformer.resblocks.0.attn.in_proj_weight', 'model.vision_tower.vision_tower.image_encoder.transformer.resblocks.0.attn.out_proj.bias', '

Initialising vision tower
Number of stacks: 1
Upsample mode: conv
tokenflow load from: /data/aofei/output/MedSight/Region_perceiver/0079280.pt
tokenflow model load success!!
mm_projector parameters unfrozen.
Loading tokenizer from /data/aofei/output/MedSight/1110_full_instruct_71k_nosep


## 1. Single-image inference

In [2]:
image_path = '/data/aofei/hallucination/VQA_RAD/images/synpic676.jpg'
question = 'What modality is used to take this image?'

answers, _ = bot.inference(question, image_path)
print(answers[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


assistant
This image is a chest X-ray, which uses ionizing radiation to capture an image of the structures within the thorax. Chest radiography is one of the most commonly used imaging modalities in clinical practice due to its low cost and ability to provide valuable information about lung diseases such as pneumonia or pulmonary edema when performed correctly.


## 2. Batch inference over VQA-RAD test split

The output file `answers_demo.jsonl` is in the same schema used by `run_eval.py`,
so you can score it with:

```bash
python run_eval.py --gt test.json --pred answers_demo.jsonl --eval_res eval_demo.txt
```

In [ ]:
import json
from tqdm import tqdm
import torch

# question_file = '/data/aofei/hallucination/VQA_RAD/data/test.json'
question_file = '/data/aofei/hallucination/Slake/data/test.json'
image_folder  = '/data/aofei/hallucination/Slake/imgs'
answers_file  = './outputs/Slake/answers_demo.jsonl'

os.makedirs(os.path.dirname(answers_file), exist_ok=True)
questions = json.load(open(question_file))

with open(answers_file, 'w') as ans_file:
    for line in tqdm(questions):
        qid       = line['id']
        question  = line['conversations'][0]['value']
        gt_answer = line['conversations'][1]['value']
        image     = os.path.join(image_folder, line['image'])

        with torch.inference_mode():
            ans = bot.inference(question, image)[0][0]
        ans = ans.replace('assistant\n', '').strip()

        ans_file.write(json.dumps({
            'question_id': qid,
            'prompt': question,
            'text': ans,
            'gt_ans': gt_answer,
            'metadata': {},
        }) + '\n')
        ans_file.flush()

100%|██████████| 451/451 [21:14<00:00,  2.83s/it]
